# 🚀 V-Netra: YOLO11n Training on Colab (Manual/Pre-built Dataset)
Notebook ini khusus digunakan jika Anda **sudah memiliki dataset ZIP** di Google Drive Anda. Proses training akan langsung dijalankan tanpa perlu mengunduh ulang dari Roboflow.


In [ ]:
!pip install -q albumentations

from google.colab import drive
import os
import shutil
import zipfile
drive.mount('/content/drive')

import os

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml fiftyone Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")

## 1. Ekstrak Dataset


In [ ]:
import os
import shutil
import zipfile


# 3. Ekstrak Dataset
master_dir = "/content/vnetra_master_dataset"
vnetra_master_dataset_zip_path = f'{INPUT_DIR}/vnetra_master_dataset.zip'
data_yaml = f'{master_dir}/data.yaml'

if not os.path.exists(master_dir):
    if os.path.exists(vnetra_master_dataset_zip_path):
        print("Mengekstrak vnetra master dataset dari Google Drive...")
        os.makedirs(master_dir, exist_ok=True)
        shutil.unpack_archive(vnetra_master_dataset_zip_path, master_dir)
        print("✅ vnetra master dataset siap digunakan.")
    else:
        print("❌ ERROR: vnetra_master_dataset.zip tidak ditemukan. Silakan jalankan Fase 1 terlebih dahulu.")
else:
    print("✅ vnetra master dataset sudah tersedia di memori.")

## 2. Pengecekan Statistik Dataset


In [ ]:
import pandas as pd
import os
import yaml

print('=== MEMBACA DATA.YAML ===')
if os.path.exists(data_yaml):
    with open(data_yaml, 'r') as f:
        yaml_data = yaml.safe_load(f)
        if isinstance(yaml_data.get('names'), dict):
            master_classes = [yaml_data['names'][i] for i in range(len(yaml_data['names']))]
        else:
            master_classes = yaml_data.get('names', [])
    print(f"📌 Ditemukan {len(master_classes)} kelas di data.yaml:")
    print(master_classes)
    
else:
    master_classes = []
    print('[ERROR] data.yaml tidak ditemukan!')

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test

    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total

    data_report.append({
        'ID': i,
        'Kelas': cls_name,
        'Train (Inst)': t_train,
        'Valid (Inst)': t_valid,
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-',
    'Kelas': 'TOTAL KESELURUHAN',
    'Train (Inst)': total_train,
    'Valid (Inst)': total_valid,
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


## 3. Training YOLO11n dengan Augmentasi OV2640
Melakukan proses pelatihan model YOLO11n dengan pengaturan hyperparameter khusus (Mosaic, rotasi, fluktuasi warna) untuk mensimulasikan tangkapan kamera OV2640 yang dipasang di dada pengguna tunanetra.


In [ ]:
from ultralytics import YOLO

# Memuat arsitektur dasar YOLO11 versi nano (Paling ringan dan cepat untuk mobile)
model = YOLO('yolo11n.pt')

    # --- [OPSIONAL] ALARM REM DARURAT SISA KUOTA ---
    # Jika sisa kuota Kaggle/Colab Anda tinggal sedikit (misal 2 Jam), aktifkan blok kode di bawah ini
    # dengan menghapus tanda pagar (#) di awal baris. YOLO akan berhenti dengan aman di jam ke-11.
    #
    # import time
    # def alarm_kuota(trainer):
    #     if time.time() - trainer.train_time_start > 5400:  # 5400 detik = 11 jam
    #         print("🚨 ALARM: Sisa kuota hampir habis! Menyimpan progress...")
    #         trainer.stop = True
    # model.add_callback("on_train_epoch_end", alarm_kuota)
    # -----------------------------------------------

results = model.train(
    # --- KONFIGURASI DATA & PERANGKAT ---
    data=f"{master_dir}/data.yaml", # Path menuju dataset yang sudah digabung
    epochs=20,                     # Maksimal putaran training (300 sudah lebih dari cukup)
    # time=11.0,                      # Otomatis Berhenti & Save dengan aman setelah 11 jam (mencegah Kaggle timeout 12 jam)
    # patience=20,                    # Otomatis berhenti jika tidak ada peningkatan mAP selama 10 epoch
    imgsz=640,                      # Resolusi standar YOLO (kamera OV2640 akan di-resize ke ukuran ini)
    batch=64,                      # Memproses 100 gambar sekaligus (Memanfaatkan RAM besar dari 2x GPU T4)
    device=0,                   # Memanfaatkan 2x GPU T4 (Multi-GPU) di Kaggle agar lebih cepat
    workers=4,                      # Optimal untuk Kaggle (memanfaatkan CPU cores untuk memuat data)
    seed=42,                        # Angka acak tetap agar hasil training bisa direproduksi/konsisten

    # --- PENYIMPANAN LOG & GRAFIK ---
    project='vnetra_training',      # Nama folder utama penyimpanan hasil
    name='yolo11n_custom',          # Nama sub-folder spesifik untuk eksperimen ini
    exist_ok=True,                  # Menimpa folder jika sudah ada (mencegah penumpukan folder eksperimen)
    # save_period=20,                 # Menyimpan file bobot cadangan setiap 20 putaran

    # --- STRATEGI PEMBELAJARAN (LEARNING) ---
    freeze=5,                       # Membekukan (tidak melatih ulang) 5 layer awal yang sudah mahir mendeteksi tepi benda (menghemat waktu)
    optimizer="AdamW",              # Lebih agresif menurunkan cls_loss dibanding SGD
    lr0=0.002,                      # Kecepatan belajar awal (tidak terlalu besar agar tidak 'nyasar', tidak terlalu kecil agar tidak lambat)
    cos_lr=True,                    # Menurunkan kecepatan belajar secara perlahan membentuk kurva kosinus (memuluskan akurasi di akhir)
    warmup_epochs=3.0,              # Pemanasan lebih panjang agar AdamW stabil dan tidak kaget di awal

    # --- RASIO KOMPROMI LOKALISASI (3 : 1) ---
    box=7.5,                        # FOKUS UTAMA: Penalti ketat agar bounding box akurat untuk perhitungan jarak
    cls=1.5,                        # FOKUS KEDUA: Insentif agar model berani menebak kelas kendaraan (Menaikkan Recall)
    dfl=1.5,                        # Mengunci ketajaman piksel di tepi rintangan (Anti-Flickering)
    label_smoothing=0.1,            # Mencegah overfitting selama maraton 10 jam

    # --- AUGMENTASI KHUSUS VNETRA (OV2640 CAMERA SIMULATION) ---
    mosaic=1.0,                     # Menggabungkan 4 gambar jadi 1, melatih model mendeteksi objek kecil dalam satu frame
    degrees=20.0,                   # Rotasi lebih natural (10 derajat) untuk kamera kepala/kacamata tanpa merusak bentuk objek
    fliplr=0.0,                     # DIMATIKAN! Jangan membalik gambar kiri-kanan, karena arah Tactile Paving (belok kiri vs kanan) bisa tertukar
    scale=0.5,                      # Skala zooming natural (YOLO default) agar objek tetap utuh

    # Simulasi kualitas gambar buruk dari kamera OV2640 (warna pudar, gelap, dll)
    hsv_h=0.015,                    # Fluktuasi hue (warna dasar)
    hsv_s=0.5,                      # Fluktuasi saturation moderat (50%) agar warna tetap wajar
    hsv_v=0.4,                      # Fluktuasi kecerahan (value) moderat (40%) meniru bayangan natural
    erasing=0.1,                    # Menghapus porsi gambar secara acak diturunkan ke 10%
)

In [ ]:
# Pindahkan bobot dan grafik hasil training ke Output Dir
import shutil
import os

best_pt_path = 'vnetra_training/yolo11n_custom/weights/best.pt'
if os.path.exists(best_pt_path):
    shutil.copy(best_pt_path, f'{OUTPUT_DIR}/best_yolo11n.pt')
    print("✅ Model Asli (.pt) berhasil disimpan ke Google Drive (Folder Output)!")

source_graph_dir = 'vnetra_training/yolo11n_custom'
target_graph_dir = f'{OUTPUT_DIR}/training_graphs'
if os.path.exists(source_graph_dir):
    if os.path.exists(target_graph_dir): shutil.rmtree(target_graph_dir)
    shutil.copytree(source_graph_dir, target_graph_dir)


## 4. Export ke LiteRT (FP32 & INT8)
Mengekspor bobot model menjadi format `.tflite` dalam bentuk kuantisasi **FP32** (Half Precision) yang sangat efisien dan kompatibel untuk *GPU Delegation* di smartphone Android.


In [ ]:
export_fp32 = model.export(format="litert", optimize=True)
import shutil
shutil.copy(export_fp32, f'{OUTPUT_DIR}/best_fp32.tflite')
print('Model FP32 berhasil disimpan ke Google Drive!')



## 5. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Test Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP32 dibanding model aslinya.

In [ ]:
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) PADA TEST SET ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map_pt = val_pt.box.map50

print("\n=== EVALUASI MODEL FP32 (.tflite) PADA TEST SET ===")
model_fp32 = YOLO(export_fp32, task='detect')
val_fp32 = model_fp32.val(data=f"{master_dir}/data.yaml", split='test')
map_fp32 = val_fp32.box.map50

print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 PADA DATASET TEST:")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP32 (.tflite)   : {map_fp32:.4f}")
print("=========================================")


## 6. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import glob

test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    res_fp32 = model_fp32.predict(source=test_img, imgsz=640)
    img_fp32 = res_fp32[0].plot()
    
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    ax[1].imshow(cv2.cvtColor(img_fp32, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model FP32 (.tflite)")
    ax[1].axis("off")
    plt.show()


## 7. Visualisasi Grafik Hasil Training
Menampilkan grafik metrik akurasi (*mAP*, *Loss*) dan *Confusion Matrix* yang telah digenerasi oleh YOLO menggunakan `matplotlib` untuk keperluan laporan skripsi.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/content/runs/detect/vnetra_training/yolo11n_custom/'
results_path = os.path.join(base_path, 'results.csv')

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('VNetra - YOLO11n Training Performance Dashboard', fontsize=26, fontweight='bold', y=0.96)
    
    sns.lineplot(data=df, x='epoch', y='train/box_loss', ax=axes[0,0], label='Train Box Loss', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='val/box_loss', ax=axes[0,0], label='Val Box Loss', linewidth=3, linestyle='--')
    axes[0,0].set_title('Box Loss Convergence', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='train/cls_loss', ax=axes[0,1], label='Train Class Loss', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='val/cls_loss', ax=axes[0,1], label='Val Class Loss', linewidth=3, linestyle='--')
    axes[0,1].set_title('Classification Loss Convergence', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50(B)', ax=axes[1,0], label='mAP@50', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='metrics/mAP50-95(B)', ax=axes[1,0], label='mAP@50-95', linewidth=3, linestyle='-.')
    axes[1,0].set_title('Mean Average Precision (mAP)', fontsize=18, fontweight='bold')
    
    sns.lineplot(data=df, x='epoch', y='metrics/precision(B)', ax=axes[1,1], label='Precision', linewidth=3)
    sns.lineplot(data=df, x='epoch', y='metrics/recall(B)', ax=axes[1,1], label='Recall', linewidth=3, linestyle=':')
    axes[1,1].set_title('Precision & Recall Trends', fontsize=18, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def display_result(image_path, width=None):
    if os.path.exists(image_path):
        if width: display(Image(filename=image_path, width=width))
        else: display(Image(filename=image_path))

print('\n=== CONFUSION MATRIX ===')
display_result(os.path.join(base_path, 'confusion_matrix_normalized.png'), width=1200)

print('\n=== KURVA F1-SCORE ===')
display_result(os.path.join(base_path, 'F1_curve.png'), width=1200)

print('\n=== AUGMENTASI MOSAIC ===')
display_result(os.path.join(base_path, 'train_batch0.jpg'), width=1200)

print('\n=== PREDIKSI PADA VALIDATION SET ===')
display_result(os.path.join(base_path, 'val_batch0_pred.jpg'), width=1200)


### 💾 5. Auto-Backup Hasil Training ke Google Drive
Seluruh metrik, grafik, dan bobot (`.pt`) akan otomatis dikompres menjadi file ZIP dan dikirim ke Google Drive agar aman dari disk *reset* Colab.

In [ ]:
import os
import shutil

print(f"\nMenge-ZIP dan membackup seluruh hasil training ke {OUTPUT_DIR}...")
# Menyimpan langsung ke variabel OUTPUT_DIR yang sudah di-set di awal notebook
shutil.make_archive(f"{OUTPUT_DIR}/vnetra_training_results", 'zip', "/content/runs/detect/vnetra_training")

print(f"\nMenyalin file model (.pt) secara langsung (tanpa di-zip) ke {OUTPUT_DIR}...")
weights_dir = "/content/runs/detect/vnetra_training/yolo11n_custom/weights"
if os.path.exists(weights_dir):
    for pt_file in ["best.pt", "last.pt"]:
        src_pt = f"{weights_dir}/{pt_file}"
        dst_pt = f"{OUTPUT_DIR}/{pt_file}"
        if os.path.exists(src_pt):
            shutil.copy2(src_pt, dst_pt)
            print(f"✔️ Berhasil menyalin {pt_file}")
        else:
            print(f"⚠️ Peringatan: {pt_file} tidak ditemukan di {weights_dir}")

print(f"\n✅ BERHASIL! Seluruh grafik & log aman di ZIP, dan file Model bisa langsung diakses di:")
print(f"📁 {OUTPUT_DIR}/")
